# Scraping des images de peintures (WGA) — à exécuter sur Google Colab

Ce notebook télécharge les images de peintures référencées dans `paintings_only.csv`
(produit par `eda_peintures.ipynb`) à partir de leurs pages sur **wga.hu**, puis
construit un sous-échantillon équilibré pour démarrer l'entraînement du GAN sans
attendre le téléchargement des 32 438 images.

**Rappel important (wga.hu/legal.html).** La Web Gallery of Art est une base de
données non commerciale maintenue par deux personnes ; ses conditions d'usage
autorisent le téléchargement d'images à des fins **éducatives et personnelles
uniquement**, et interdisent toute **redistribution**. Ce notebook :
- limite volontairement la cadence et la concurrence des requêtes (site modeste,
  pas d'infrastructure de scraping à grande échelle),
- ne télécharge que pour un usage local d'expérimentation (entraînement GAN
  personnel) — **ne republie pas** les images en l'état.

**Plan.**
1. Préparer l'environnement
2. Récupérer `paintings_only.csv` (Drive ou upload direct)
3. Déployer le script de scraping (`scrape_images.py`)
4. Déployer le script d'échantillonnage équilibré (`build_subset.py`)
5. Construire le sous-échantillon équilibré
6. Test rapide du scraper (politesse : valider avant de lancer en masse)
7. Lancement complet du sous-échantillon
8. Vérification visuelle + sauvegarde


## 1. Préparer l'environnement

In [ ]:
import os
print("Environnement :", "Google Colab" if "COLAB_RELEASE_TAG" in os.environ else "local / autre")
!pip install -q requests pandas tqdm


## 2. Récupérer `paintings_only.csv`

Deux options :
- **Google Drive (recommandé)** : les images et fichiers intermédiaires survivent
  à une déconnexion de session Colab, et le scraping peut reprendre là où il s'est
  arrêté (le script ignore les fichiers déjà téléchargés).
- **Upload direct** : plus simple pour un essai ponctuel, mais tout est perdu si la
  session Colab se déconnecte.

Mets `USE_DRIVE = False` dans la cellule suivante si tu préfères l'upload direct.


In [ ]:
from pathlib import Path

USE_DRIVE = True  # False pour tout garder dans le stockage éphémère de Colab

BASE = Path("/content/GAN")
BASE.mkdir(parents=True, exist_ok=True)
%cd $BASE

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/GAN")
else:
    DATA_DIR = BASE / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

csv_path = DATA_DIR / "paintings_only.csv"
print("Fichier attendu :", csv_path, "-> présent :", csv_path.exists())


In [ ]:
if not csv_path.exists():
    from google.colab import files
    print("Uploade ici le fichier data/paintings_only.csv (produit par eda_peintures.ipynb) :")
    uploaded = files.upload()
    for name, content in uploaded.items():
        (DATA_DIR / name).write_bytes(content)
    csv_path = DATA_DIR / "paintings_only.csv"

assert csv_path.exists(), "paintings_only.csv introuvable — uploade-le ou vérifie DATA_DIR."
print("OK :", csv_path)


## 3. Déployer le script de scraping

Mapping découvert en inspectant une page wga.hu : une page HTML
`https://www.wga.hu/html/{a}/{artiste}/{oeuvre}.html` a toujours son image pleine
résolution à `https://www.wga.hu/art/{a}/{artiste}/{oeuvre}.jpg`. Le script
télécharge en parallèle limité (par défaut 4 workers), avec délai, retries et
reprise automatique (il ignore un fichier déjà présent sur disque).


In [ ]:
%%writefile scrape_images.py
"""
Télécharge les images des peintures depuis les pages wga.hu référencées dans
data/paintings_only.csv (colonne URL).

Mapping observé sur le site : une page HTML
    https://www.wga.hu/html/{a}/{artiste}/{oeuvre}.html
a toujours son image pleine résolution à
    https://www.wga.hu/art/{a}/{artiste}/{oeuvre}.jpg

"""


import argparse
import csv
import time
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

USER_AGENT = "Mozilla/5.0 (compatible; EDA-research-script/1.0; personal/educational GAN dataset project)"


def html_url_to_image_url(html_url: str) -> str:
    return html_url.replace("/html/", "/art/").replace(".html", ".jpg")


def target_path(html_url: str, out_dir: Path) -> Path:
    # mirrors the /a/artiste/oeuvre.jpg structure sous out_dir/, pour garantir
    # des noms de fichiers uniques et pouvoir retrouver la ligne CSV d'origine.
    tail = html_url.split("/html/", 1)[1]  # ex: a/aachen/adonis.html
    return out_dir / Path(tail).with_suffix(".jpg")


def download_one(session, html_url, out_dir, timeout, min_delay, max_retries) -> tuple:
    img_url = html_url_to_image_url(html_url)
    dest = target_path(html_url, out_dir)

    if dest.exists() and dest.stat().st_size > 0:
        return html_url, "skip (déjà présent)"

    dest.parent.mkdir(parents=True, exist_ok=True)

    last_err = ""
    for attempt in range(1, max_retries + 1):
        try:
            time.sleep(min_delay + random.uniform(0, 0.2))
            resp = session.get(img_url, timeout=timeout)
            if resp.status_code == 200 and resp.content:
                dest.write_bytes(resp.content)
                return html_url, "ok"
            last_err = f"http {resp.status_code}"
        except requests.RequestException as e:
            last_err = str(e)
        time.sleep(1.5 * attempt)  # backoff
    return html_url, f"échec ({last_err})"


def scrape(csv_path, out_dir, limit=None, workers=4, min_delay=0.4, timeout=20, max_retries=3):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    failed_log = out_dir / "_failed.csv"

    with open(csv_path, encoding="utf-8") as f:
        urls = [row["URL"] for row in csv.DictReader(f)]
    if limit:
        urls = urls[:limit]

    print(f"{len(urls):,} images à traiter (concurrence = {workers})")

    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    ok = skip = fail = 0
    failures = []
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {
            ex.submit(download_one, session, u, out_dir, timeout, min_delay, max_retries): u
            for u in urls
        }
        for i, fut in enumerate(as_completed(futures), 1):
            url, status = fut.result()
            if status == "ok":
                ok += 1
            elif status.startswith("skip"):
                skip += 1
            else:
                fail += 1
                failures.append((url, status))

            if i % 200 == 0 or i == len(urls):
                elapsed = time.time() - t0
                rate = i / elapsed if elapsed > 0 else 0
                print(f"  {i:,}/{len(urls):,}  ok={ok} skip={skip} fail={fail}  "
                      f"({rate:.1f} img/s, {elapsed/60:.1f} min écoulées)")

    if failures:
        with open(failed_log, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["URL", "erreur"])
            w.writerows(failures)
        print(f"\n{len(failures)} échecs loggés dans {failed_log}")

    print(f"\nTerminé en {(time.time()-t0)/60:.1f} min — ok={ok} skip={skip} fail={fail}")
    return ok, skip, fail


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="data/paintings_only.csv", help="CSV source (colonne URL)")
    ap.add_argument("--out", default="images", help="dossier de destination des images")
    ap.add_argument("--limit", type=int, default=None, help="ne traiter que les N premières lignes (test)")
    ap.add_argument("--workers", type=int, default=4, help="requêtes en parallèle")
    ap.add_argument("--min-delay", type=float, default=0.4, help="délai mini entre deux requêtes par worker (s)")
    ap.add_argument("--timeout", type=float, default=20, help="timeout réseau par requête (s)")
    ap.add_argument("--max-retries", type=int, default=3)
    args = ap.parse_args()

    scrape(args.csv, args.out, args.limit, args.workers, args.min_delay, args.timeout, args.max_retries)


if __name__ == "__main__":
    main()


## 4. Déployer le script d'échantillonnage équilibré

L'EDA (`eda_peintures.ipynb`, section 12) a montré un corpus très déséquilibré :
42&nbsp;% de peinture religieuse, 42&nbsp;% d'école italienne. `build_subset.py`
tire un sous-échantillon à quota égal par genre (`TYPE`) puis, à l'intérieur de
chaque genre, à quota égal par école (`SCHOOL`) dans la limite du disponible.


In [ ]:
%%writefile build_subset.py
"""
Construit un sous-échantillon équilibré de data/paintings_only.csv pour démarrer
rapidement l'entraînement du GAN, sans attendre le scraping complet des 32 438
images (voir eda_peintures.ipynb, section 12 : le corpus brut est dominé à 42%
par le religieux et 42% par l'école italienne).

Stratégie :
  1. Quota égal par TYPE (ex. 4000 / 10 genres = 400 par genre) -> corrige le
     déséquilibre de genre.
  2. À l'intérieur de chaque genre, quota égal par SCHOOL présente -> corrige
     en partie le déséquilibre d'école, dans la limite du disponible.
  3. Les quotas non atteints (écoles trop petites) sont redistribués
     aléatoirement sur le reste du genre pour atteindre la cible.

Usage :
    python build_subset.py --size 4000
"""
import argparse
import pandas as pd

SEED = 42


def stratified_sample(paint: pd.DataFrame, target_total: int) -> pd.DataFrame:
    types = paint["TYPE"].unique()
    quota_per_type = max(1, target_total // len(types))

    picked = []
    for t in types:
        sub = paint[paint["TYPE"] == t]
        schools = sub["SCHOOL"].unique()
        per_school = max(1, quota_per_type // len(schools))

        chosen = sub.groupby("SCHOOL", group_keys=False).apply(
            lambda g: g.sample(min(len(g), per_school), random_state=SEED)
        )

        # complète jusqu'au quota du genre avec le reste disponible (écoles
        # trop petites pour fournir leur part)
        missing = quota_per_type - len(chosen)
        if missing > 0:
            remaining = sub.drop(chosen.index)
            top_up = remaining.sample(min(missing, len(remaining)), random_state=SEED)
            chosen = pd.concat([chosen, top_up])

        picked.append(chosen)

    return pd.concat(picked).sample(frac=1, random_state=SEED)  # mélange final


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--size", type=int, default=4000, help="taille cible du sous-échantillon")
    ap.add_argument("--src", default="data/paintings_only.csv")
    ap.add_argument("--out", default="data/subset_sample.csv")
    args = ap.parse_args()

    paint = pd.read_csv(args.src)
    subset = stratified_sample(paint, args.size)
    subset.to_csv(args.out, index=False)

    def dist(df, col):
        return (df[col].value_counts(normalize=True) * 100).round(1)

    print(f"Sous-échantillon : {len(subset):,} lignes -> {args.out}")
    print("\nTYPE — corpus complet vs sous-échantillon (%) :")
    print(pd.concat([dist(paint, "TYPE"), dist(subset, "TYPE")], axis=1,
                     keys=["complet", "sous-éch."]).fillna(0))
    print("\nSCHOOL (top 8) — corpus complet vs sous-échantillon (%) :")
    top_schools = paint["SCHOOL"].value_counts().head(8).index
    print(pd.concat([dist(paint, "SCHOOL"), dist(subset, "SCHOOL")], axis=1,
                     keys=["complet", "sous-éch."]).reindex(top_schools).fillna(0))


if __name__ == "__main__":
    main()


## 5. Construire le sous-échantillon équilibré (par défaut : 4 000 images)

In [ ]:
subset_csv = DATA_DIR / "subset_sample.csv"
!python build_subset.py --src "{csv_path}" --out "{subset_csv}" --size 4000


## 6. Test rapide du scraper

Avant de lancer ~4 000 téléchargements, on valide sur un tout petit échantillon
(20 images) que le mapping d'URL et le réseau fonctionnent depuis Colab.


In [ ]:
IMAGES_DIR = DATA_DIR / "images"
!python scrape_images.py --csv "{subset_csv}" --out "{IMAGES_DIR}" --limit 20 --workers 3 --min-delay 0.6 --timeout 25


## 7. Lancement complet du sous-échantillon

⚠️ Cette cellule peut prendre **1 à 3 heures** selon la charge du site. Si la
session Colab se déconnecte, relance simplement la même cellule : les images déjà
téléchargées sont automatiquement ignorées (`skip`), donc le travail déjà fait
n'est jamais perdu (surtout utile avec `USE_DRIVE = True`).


In [ ]:
!python scrape_images.py --csv "{subset_csv}" --out "{IMAGES_DIR}" --workers 3 --min-delay 0.6 --timeout 25


## 8. Vérification visuelle et sauvegarde

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

files_list = list(IMAGES_DIR.rglob("*.jpg"))
print(f"{len(files_list):,} images présentes dans {IMAGES_DIR}")

sample = random.sample(files_list, min(4, len(files_list)))
fig, axes = plt.subplots(1, len(sample), figsize=(3.2 * len(sample), 3.6))
axes = [axes] if len(sample) == 1 else axes
for ax, p in zip(axes, sample):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
if not USE_DRIVE:
    import shutil
    from google.colab import files as colab_files
    archive = shutil.make_archive("/content/images_subset", "zip", IMAGES_DIR)
    colab_files.download(archive)
else:
    print("Images déjà persistées sur Google Drive :", IMAGES_DIR)


---
**Rappel.** Ces images restent destinées à un usage personnel/éducatif (entraînement
du GAN en local/Colab) — ne pas les republier ni les redistribuer, conformément aux
conditions d'utilisation de la Web Gallery of Art.
